In [1]:
import pandas as pd
import json
import sys
import os
from pathlib import Path

# Project root
PROJECT_ROOT = Path.cwd().parent
# Add project root to Python path
sys.path.insert(0, str(PROJECT_ROOT))

from src.data.financial_data_loader import (FinancialDataLoader)
from src.tools.tool_executor import (
    ToolExecutor
)

In [2]:
user_agent = os.getenv(
    "SEC_USER_AGENT"
)

loader = FinancialDataLoader(
    user_agent
)

In [8]:
financials = loader.load("Apple")

In [9]:
financials.head()

,fiscal_year,revenue,net_income,assets,liabilities,cash,operating_cash_flow
0,2017,229.234,48.351,375.319,241.272,20.289,64.225
1,2018,265.595,59.531,365.725,258.578,25.913,77.434
2,2019,260.174,55.256,338.516,248.028,48.844,69.391
3,2020,274.515,57.411,323.888,258.549,38.016,80.674
4,2021,365.817,94.680,351.002,287.912,34.940,104.038


In [10]:
amazon = loader.load("Amazon")

In [11]:
amazon.head()

,fiscal_year,revenue,net_income,assets,cash,operating_cash_flow
0,2006,NaN,NaN,NaN,1.022,NaN
1,2007,NaN,0.476,NaN,2.539,1.405
2,2008,NaN,0.645,8.314,2.769,1.697
3,2009,NaN,0.902,13.813,3.444,3.293
4,2010,NaN,1.152,18.797,3.777,3.495


In [12]:
amazon.columns

Index(['fiscal_year', 'revenue', 'net_income', 'assets', 'cash',
       'operating_cash_flow'],
      dtype='str')

### Create the executor

In [14]:
executor = ToolExecutor(financials)

### Execute a tool

In [16]:
result = executor.execute(
    "get_revenue_growth"
)

result

,fiscal_year,revenue,revenue_growth_pct
0,2017,229.234,NaN
1,2018,265.595,15.861958
2,2019,260.174,-2.041078
3,2020,274.515,5.512080
4,2021,365.817,33.259385
5,2022,394.328,7.793788
6,2023,383.285,-2.800461
7,2024,391.035,2.021994
8,2025,416.161,6.425512


In [19]:
profit = executor.execute(
    "get_profit_margin"
)

profit

,fiscal_year,revenue,net_income,net_profit_margin_pct
0,2017,229.234,48.351,21.092421
1,2018,265.595,59.531,22.414202
2,2019,260.174,55.256,21.238095
3,2020,274.515,57.411,20.913611
4,2021,365.817,94.680,25.881793
5,2022,394.328,99.803,25.309641
6,2023,383.285,96.995,25.306234
7,2024,391.035,93.736,23.971256
8,2025,416.161,112.010,26.915064


In [20]:
summary = executor.execute(
    "get_latest_summary"
)

summary

{'fiscal_year': 2025,
 'revenue_billions': 416.161,
 'net_income_billions': 112.01,
 'assets_billions': 359.241,
 'liabilities_billions': 285.508,
 'cash_billions': 35.934,
 'operating_cash_flow_billions': 111.482,
 'revenue_growth_pct': 6.425511782832727,
 'net_profit_margin_pct': 26.91506412181824,
 'operating_cash_flow_margin_pct': 26.788190147563085,
 'liability_to_asset_pct': 79.47533828265705}

### Test a company

In [24]:
amazon_financials = loader.load(
    "Amazon"
)

In [25]:
amazon_executor = ToolExecutor(
    amazon_financials
)

In [26]:
amazon_executor.execute(
    "get_revenue_growth"
)

,fiscal_year,revenue,revenue_growth_pct
0,2006,NaN,NaN
1,2007,NaN,NaN
2,2008,NaN,NaN
3,2009,NaN,NaN
4,2010,NaN,NaN
5,2011,NaN,NaN
6,2012,NaN,NaN
7,2013,NaN,NaN
8,2014,NaN,NaN
9,2015,NaN,NaN


### Assistant Exploration end to end flow from user, router, executor

In [3]:
from src.application.financial_assistant import FinancialAssistant

assistant = FinancialAssistant()

DEBUG A: Starting tokenizer


DEBUG B: Tokenizer loaded
DEBUG C: Starting model


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

DEBUG D: Model loaded


In [3]:
result = assistant.ask(
    "How has Apple's revenue grown?"
)

result

{'question': "How has Apple's revenue grown?",
 'company': 'Apple',
 'tool': 'get_revenue_growth',
 'result':    fiscal_year  revenue  revenue_growth_pct
 0         2017  229.234                 NaN
 1         2018  265.595           15.861958
 2         2019  260.174           -2.041078
 3         2020  274.515            5.512080
 4         2021  365.817           33.259385
 5         2022  394.328            7.793788
 6         2023  383.285           -2.800461
 7         2024  391.035            2.021994
 8         2025  416.161            6.425512}

### Test different questions

In [4]:
result2= assistant.ask(
    "Is Apple's profitability improving?"
)

result2

{'question': "Is Apple's profitability improving?",
 'company': 'Apple',
 'tool': 'get_profit_margin',
 'result':    fiscal_year  revenue  net_income  net_profit_margin_pct
 0         2017  229.234      48.351              21.092421
 1         2018  265.595      59.531              22.414202
 2         2019  260.174      55.256              21.238095
 3         2020  274.515      57.411              20.913611
 4         2021  365.817      94.680              25.881793
 5         2022  394.328      99.803              25.309641
 6         2023  383.285      96.995              25.306234
 7         2024  391.035      93.736              23.971256
 8         2025  416.161     112.010              26.915064}

In [5]:
result3 = assistant.ask(
    "How quickly are Apple's assets growing?"
)
result3

{'question': "How quickly are Apple's assets growing?",
 'company': 'Apple',
 'tool': 'get_asset_growth',
 'result':    fiscal_year   assets  asset_growth_pct
 0         2017  375.319               NaN
 1         2018  365.725         -2.556226
 2         2019  338.516         -7.439743
 3         2020  323.888         -4.321214
 4         2021  351.002          8.371412
 5         2022  352.755          0.499427
 6         2023  352.583         -0.048759
 7         2024  364.980          3.516052
 8         2025  359.241         -1.572415}

In [6]:
result4 = assistant.ask(
    "What is Apple's long-term revenue growth rate?"
)
result4

{'question': "What is Apple's long-term revenue growth rate?",
 'company': 'Apple',
 'tool': 'get_revenue_cagr',
 'result': np.float64(7.738963527421405)}

In [7]:
result5 = assistant.ask(
    "Is Apple's net income growing faster than its revenue?"
)
result5

{'question': "Is Apple's net income growing faster than its revenue?",
 'company': 'Apple',
 'tool': 'get_net_income_cagr',
 'result': np.float64(11.072466771129319)}

### Test another company

In [8]:
result6 = assistant.ask(
    "How has Amazons's revenue grown?"
)
result6

{'question': "How has Amazons's revenue grown?",
 'company': 'Amazon',
 'tool': 'get_revenue_growth',
 'result':     fiscal_year  revenue  revenue_growth_pct
 0          2006      NaN                 NaN
 1          2007      NaN                 NaN
 2          2008      NaN                 NaN
 3          2009      NaN                 NaN
 4          2010      NaN                 NaN
 5          2011      NaN                 NaN
 6          2012      NaN                 NaN
 7          2013      NaN                 NaN
 8          2014      NaN                 NaN
 9          2015      NaN                 NaN
 10         2016  135.987                 NaN
 11         2017  177.866           30.796326
 12         2018  232.887           30.933962
 13         2019  280.522           20.454126
 14         2020  386.064           37.623431
 15         2021  469.822           21.695367
 16         2022  513.983            9.399517
 17         2023  574.785           11.829574
 18         20

### Test the AnswerGenerator separately

In [ ]:
from src.llm.answer_generator import AnswerGenerator

answer_generator = AnswerGenerator()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

: 

In [6]:
result = assistant.ask(
    "How has Apple's revenue grown?"
)

In [7]:
answer = answer_generator.generate(
    question=result["question"],
    company=result["company"],
    tool_name=result["tool"],
    result=result["result"]
)

print(answer)

Apple's revenue has grown at an average rate of approximately 15.9% per year since 2017. This indicates that Apple's revenue has been increasing steadily over this period.


### Test different types of results

In [8]:
result = assistant.ask(
    "What is Apple's long-term revenue growth rate?"
)

answer = answer_generator.generate(
    question=result["question"],
    company=result["company"],
    tool_name=result["tool"],
    result=result["result"]
)

print(answer)

Apple's long-term revenue growth rate is 7.74%.


In [9]:
result = assistant.ask(
    "What is Apple's long-term revenue growth rate?"
)

answer = answer_generator.generate(
    question=result["question"],
    company=result["company"],
    tool_name=result["tool"],
    result=result["result"]
)

print(answer)

Apple's long-term revenue growth rate is 7.74%.


#### Test another company

In [ ]:
result = assistant.ask(
    "How has Amazon's revenue grown?"
)

answer = answer_generator.generate(
    question=result["question"],
    company=result["company"],
    tool_name=result["tool"],
    result=result["result"]
)

print(answer)

Amazon's revenue has grown significantly over the years. From 2006 to 2025, Amazon's revenue grew at an average rate of approximately 17% per year. This growth can be attributed to several factors including increased sales volume, improved product offerings, and expansion into new markets. The highest revenue growth occurred from 2012 to 2013, with annual revenue growth rates of about 30%. However, the revenue growth slowed down after that period, indicating potential challenges ahead for Amazon in its competitive landscape.


### new prompt for better results 

In [4]:
from src.llm.answer_generator import AnswerGenerator

answer_generator = AnswerGenerator(llm=assistant.llm)

In [6]:
result = assistant.ask(
    "What is Apple's long-term revenue growth rate?"
)

print(result["answer"])

Apple's long-term revenue growth rate is 7.74%. This represents the compound annual growth rate of revenue over the available period.


In [7]:
result = assistant.ask(
    "How has Amazon's revenue grown?"
)

print(result["answer"])

Amazon's revenue grew at an average rate of 21.7% per year from 2016 to 2025, with fluctuations throughout the period.


In [8]:
result = assistant.ask(
    "Is Apple's profitability improving?"
)

print(result["answer"])

Apple's profitability is improving steadily over the years. The net profit margin has been increasing at an average rate of 2.1% per year since 2017.


In [9]:
result = assistant.ask(
    "How has Apple's revenue grown?"
)

print(result["answer"])

Apple's revenue has grown at an average rate of 15.86% per year since 2017.


In [10]:
result = assistant.ask(
    "Is Apple's net income growing faster than its revenue?"
)

print(result["answer"])

Yes, Apple's net income is growing faster than its revenue. The net income CAGR (Compound Annual Growth Rate) for Apple is 11.07%, indicating an increasing trend in its profitability.


In [5]:
questions = [
    "What is Apple's long-term revenue growth rate?",
    "How has Amazon's revenue grown?",
    "Is Apple's profitability improving?",
    "How has Apple's revenue grown?"
]

for question in questions:
    result = assistant.ask(question)
    print("\nQUESTION:", question)
    print("ANSWER:", result["answer"])


QUESTION: What is Apple's long-term revenue growth rate?
ANSWER: Apple's long-term revenue growth rate is 7.74% per year.

QUESTION: How has Amazon's revenue grown?
ANSWER: Amazon's revenue grew from $135.99 billion in 2016 to $637.96 billion in 2024, with an overall increase of approximately 376% over the 8-year period.

QUESTION: Is Apple's profitability improving?
ANSWER: Yes, Apple's profitability is generally improving over time, with higher net profit margins consistently showing an increase from 2017 to 2025.

QUESTION: How has Apple's revenue grown?
ANSWER: Apple's revenue grew from $229.23 billion in 2017 to $394.33 billion in 2024, marking an overall increase of 69.4%.


### With answer validator

In [5]:
result = assistant.ask(
    "Is Apple's profitability improving?"
)

result

{'question': "Is Apple's profitability improving?",
 'company': 'Apple',
 'tool': 'get_profit_margin',
 'result':    fiscal_year  revenue  net_income  net_profit_margin_pct
 0         2017  229.234      48.351              21.092421
 1         2018  265.595      59.531              22.414202
 2         2019  260.174      55.256              21.238095
 3         2020  274.515      57.411              20.913611
 4         2021  365.817      94.680              25.881793
 5         2022  394.328      99.803              25.309641
 6         2023  383.285      96.995              25.306234
 7         2024  391.035      93.736              23.971256
 8         2025  416.161     112.010              26.915064,
 'answer': "Yes, Apple's profitability is improving over time. The company's net profit margin increased from 21.09% in 2017 to 25.88% in 2024. This indicates better efficiency in managing costs relative to its revenue.",
 'answer_valid': True}

In [6]:
from src.llm.answer_validator import AnswerValidator

validator = AnswerValidator()

bad_answer = "Apple's profitability has consistently increased."

validator.validate(
    "get_profit_margin",
    result["result"],
    bad_answer
)

False

In [7]:
good_answer = (
    "Apple's profitability fluctuated over the period, "
    "with the net profit margin increasing from 21.09% "
    "in 2017 to 25.88% in 2025."
)

validator.validate(
    "get_profit_margin",
    result["result"],
    good_answer
)

True

### new retry validator

In [5]:
result = assistant.ask(
    "Is Apple's profitability improving?"
)

print(result["answer"])
print(result["answer_valid"])

Answer validation attempt 1: False
Answer validation attempt 2: True
Yes, Apple's profitability (net profit margin) has improved slightly over time, ranging from 21.09% in 2017 to 26.92% in 2025. However, there have been periods when the margin declined, indicating mixed performance.
True
